# MEV / Limit Book — Liquidity Surface & Arb Context

Demo the **order-book volume** and **MEV/arb** tables landed by
`ccquant sync depth` / `sync mev` + dbt marts.

**What this measures**

| Layer | Question it answers |
|---|---|
| CEX depth (bps bands) | How much **notional** sits within 10/25/50 bps of mid? Is the book **skewed** (imbalance)? |
| CEX–DEX basis | Is CEX mid rich/cheap vs a free DEX/oracle price (arb capacity)? |
| MEV-Boost (optional) | What was ETH PBS **block value** that day (date-global context)? |

**Not in v1:** sandwich catalogs, EigenPhi labels, or joins into `mart_signals_daily`.

**Prereqs**
```bash
uv run ccquant sync depth --top 20
uv run ccquant sync mev --top 20
uv run dbt build --select +mart_mev_arb_daily --project-dir dbt --profiles-dir dbt
```
Optional MEV-Boost: drop parquet under `data/mev/mevboost` then `ccquant import mev-boost --source data/mev/mevboost`.


In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import duckdb
import plotly.express as px
import polars as pl
from dotenv import load_dotenv
from IPython.display import display

from ccquant.forecasting import load_mev_panel, load_order_book_panel

warnings.filterwarnings("ignore", category=FutureWarning)

_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
load_dotenv(_root / ".env", override=True)

DB = Path(os.environ.get("CCQUANT_DB", _root / "data" / "ccquant.duckdb")).resolve()
print(f"DB: {DB}  exists={DB.exists()}")

## 1. Inventory — what landed in DuckDB

Raw tables are Python-owned; marts need `dbt build`. Empty MEV-Boost is expected until you import parquet.

In [ ]:
def _count(con: duckdb.DuckDBPyConnection, sql: str) -> int:
    try:
        return int(con.execute(sql).fetchone()[0])
    except Exception:
        return -1


with duckdb.connect(str(DB), read_only=True) as con:
    inv = pl.DataFrame(
        {
            "table": [
                "main.order_book_snapshots",
                "main.dex_price_daily",
                "main.mev_boost_payloads",
                "main_marts.fct_order_book_agg",
                "main_marts.fct_cex_dex_basis",
                "main_marts.mart_mev_arb_daily",
            ],
            "rows": [
                _count(con, "select count(*) from main.order_book_snapshots"),
                _count(con, "select count(*) from main.dex_price_daily"),
                _count(con, "select count(*) from main.mev_boost_payloads"),
                _count(con, "select count(*) from main_marts.fct_order_book_agg"),
                _count(con, "select count(*) from main_marts.fct_cex_dex_basis"),
                _count(con, "select count(*) from main_marts.mart_mev_arb_daily"),
            ],
        }
    )
    by_ex = con.execute(
        """
        select exchange, count(*) as snapshots, count(distinct symbol) as symbols
        from main.order_book_snapshots
        group by 1 order by 2 desc
        """
    ).pl()

display(inv)
display(by_ex)
print(
    "Value so far: live CEX depth snapshots (forward-only) + DefiLlama DEX prices."
    " MEV-Boost stays 0 until parquet is imported."
)

## 2. Order-book volume — where is liquidity?

`total_notional_bps_25` ≈ USD notional of bids+asks within **±25 bps of mid**.
That is the free-data proxy for “order book volume” usable for arb capacity
(how much you can lift without walking far through the book).

`imbalance_bps_25` ∈ [-1, 1]: positive → more bid notional in-band (buy support).

In [ ]:
ob = load_order_book_panel(DB)
if ob.is_empty():
    raise RuntimeError(
        "fct_order_book_agg missing/empty — run sync depth + "
        "dbt build --select +mart_mev_arb_daily"
    )

latest = (
    ob.sort("timestamp")
    .group_by("symbol")
    .last()
    .with_columns(
        (pl.col("total_notional_bps_25") / 1e6).alias("depth_25bps_musd"),
        (pl.col("bid_notional_bps_25") / 1e6).alias("bid_25_musd"),
        (pl.col("ask_notional_bps_25") / 1e6).alias("ask_25_musd"),
    )
    .sort("depth_25bps_musd", descending=True)
)

show = latest.select(
    "symbol",
    "timestamp",
    "exchange_count",
    "mid",
    "spread_bps",
    "depth_25bps_musd",
    "bid_25_musd",
    "ask_25_musd",
    "imbalance_bps_25",
)
display(show)

focus = ["BTC", "ETH", "SOL", "BNB", "XRP"]
focus_df = show.filter(pl.col("symbol").is_in(focus))
if not focus_df.is_empty():
    print("Majors (latest snapshot):")
    display(focus_df)

In [ ]:
pdf = show.to_pandas()
fig = px.bar(
    pdf.head(15),
    x="symbol",
    y="depth_25bps_musd",
    color="imbalance_bps_25",
    color_continuous_scale="RdYlGn",
    title="Near-touch liquidity: ±25 bps notional (USD millions)",
    labels={
        "depth_25bps_musd": "Bid+Ask notional within ±25 bps ($M)",
        "imbalance_bps_25": "Imbalance",
    },
    template="plotly_dark",
)
fig.update_layout(height=420, margin=dict(l=40, r=20, t=50, b=40))
fig.show()

top = pdf.iloc[0]
print(
    f"Deepest book in this poll: {top['symbol']} — "
    f"~${top['depth_25bps_musd']:.2f}M within ±25 bps, "
    f"spread {top['spread_bps']:.3f} bps, "
    f"imbalance {top['imbalance_bps_25']:+.3f}."
)

## 3. CEX–DEX basis — arb signal (when dates align)

`basis_bps = (cex_mid_last - dex_price) / cex_mid_last × 10_000`

- Positive → CEX rich vs DefiLlama (sell CEX / buy DEX idea, conceptually)
- Negative → CEX cheap

Join is on `symbol × date`. If depth was polled near UTC midnight and DEX
prices stamp the next UTC day, `basis_bps` can be null until both share a date
(re-poll depth + `sync mev` on the same calendar day, then rebuild).

In [ ]:
with duckdb.connect(str(DB), read_only=True) as con:
    basis = con.execute(
        """
        select symbol, date, venue, dex_price_usd, cex_mid_last,
               avg_spread_bps, avg_depth_notional_bps_25 / 1e6 as depth_25_musd,
               snapshot_count, basis_bps
        from main_marts.fct_cex_dex_basis
        order by abs(coalesce(basis_bps, 0)) desc, symbol
        """
    ).pl()

display(basis.head(20))
n_joined = basis.filter(pl.col("basis_bps").is_not_null()).height
print(f"Rows with computable basis_bps: {n_joined} / {basis.height}")

if n_joined:
    bpdf = basis.filter(pl.col("basis_bps").is_not_null()).to_pandas()
    fig = px.bar(
        bpdf.head(15),
        x="symbol",
        y="basis_bps",
        title="CEX mid vs DefiLlama — basis (bps)",
        template="plotly_dark",
        color="basis_bps",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print(
        "No overlapping symbol×date yet. Typical fix:\n"
        "  uv run ccquant sync depth --top 20 && uv run ccquant sync mev --top 20\n"
        "  uv run dbt build --select +mart_mev_arb_daily --project-dir dbt --profiles-dir dbt"
    )

## 4. Daily MEV/arb panel

`load_mev_panel()` → `mart_mev_arb_daily` (depth rollup + basis + optional boost).
**MEV-Boost columns are date-global ETH PBS totals** — same value on every symbol
for a date when dumps are imported; null if you have not imported parquet.

In [ ]:
mev = load_mev_panel(DB)
if mev.is_empty():
    print("mart_mev_arb_daily missing — run dbt build --select +mart_mev_arb_daily")
else:
    cov = {
        "rows": mev.height,
        "symbols": mev["symbol"].n_unique(),
        "with_depth": mev.filter(pl.col("avg_depth_notional_bps_25").is_not_null()).height,
        "with_basis": mev.filter(pl.col("basis_bps").is_not_null()).height,
        "with_boost": mev.filter(pl.col("mev_boost_value_eth").is_not_null()).height,
    }
    print("Coverage:", cov)
    display(
        mev.sort("avg_depth_notional_bps_25", descending=True, nulls_last=True).head(15)
    )

    majors = mev.filter(pl.col("symbol").is_in(["BTC", "ETH", "SOL"]))
    if not majors.is_empty():
        print("BTC / ETH / SOL panel rows:")
        display(majors.sort(["symbol", "date"]))

## 5. Why this is useful (quant read)

1. **Liquidity screen** — Rank symbols by ±25 bps notional before sizing a
   cross-venue or CEX↔DEX idea; thin books (low `$M`) are expensive to arb.
2. **Microstructure tilt** — Persistent positive/negative `imbalance_bps_25`
   with tight `spread_bps` is a short-horizon pressure cue (not a long-horizon alpha by itself).
3. **Basis + depth together** — A large `|basis_bps|` only matters if
   `depth_25bps_musd` can absorb the trade; otherwise the “arb” is illusory.
4. **MEV-Boost context** — When imported, high PBS value days mark competitive
   ETH block space; useful as a **regime flag**, not a per-token feature.

### Caveats

- Depth is **forward-only** (no free multi-year L2 history).
- One REST poll ≠ continuous book; treat as sparse snapshots.
- DefiLlama price ≠ executable DEX mid on every venue.
- Close notebook kernels before `dbt build` / write syncs (DuckDB single-writer).

In [ ]:
# One-glance snapshot for the current DB
if not latest.is_empty():
    deep = latest.head(3)
    tight = latest.sort("spread_bps").head(3)
    print("Top depth (±25 bps $M):")
    for r in deep.iter_rows(named=True):
        print(
            f"  {r['symbol']:6}  depth={r['depth_25bps_musd']:8.2f} M  "
            f"spread={r['spread_bps']:6.3f} bps  imb={r['imbalance_bps_25']:+.3f}"
        )
    print("Tightest spreads:")
    for r in tight.iter_rows(named=True):
        print(
            f"  {r['symbol']:6}  spread={r['spread_bps']:6.3f} bps  "
            f"mid={r['mid']}"
        )